In [ ]:
import sys
import pandas as pd
from pathlib import Path

sys.path.append(str(Path("..").resolve()))
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

from etl.extract import load_datasets
from etl.transform import optimize_dataframe, transform_customers, transform_inventory, transform_orders, transform_products, transform_promotions, transform_reviews
from etl.load import save_dataframe_as_csv, save_dataframe_as_parquet

#CARGA DE DATASETS y optimización
datasets = load_datasets(Path("../data")) 

for file_name, df in datasets.items():
    datasets[file_name] = optimize_dataframe(df, name = file_name)

#Solo intenta usar el tipo de dato mas chico posible sin romper los datos. Optimización de memoria entre 50%-80%


--- Inicio de Optimización en archivo: ecommerce_brands ---
📊 Tamaño previo: 0.00 MB | Filas: 15
✅ Ejecución exitosa.
📊 Tamaño optimizado: 0.00 MB
📉 Reducción de memoria: 20.78%
--- Inicio de Optimización en archivo: ecommerce_categories ---
📊 Tamaño previo: 0.00 MB | Filas: 10
✅ Ejecución exitosa.
📊 Tamaño optimizado: 0.00 MB
📉 Reducción de memoria: 6.57%
--- Inicio de Optimización en archivo: ecommerce_customers ---
📊 Tamaño previo: 0.21 MB | Filas: 334
✅ Ejecución exitosa.
📊 Tamaño optimizado: 0.12 MB
📉 Reducción de memoria: 42.61%
--- Inicio de Optimización en archivo: ecommerce_inventory ---
📊 Tamaño previo: 0.02 MB | Filas: 187
✅ Ejecución exitosa.
📊 Tamaño optimizado: 0.01 MB
📉 Reducción de memoria: 36.21%
--- Inicio de Optimización en archivo: ecommerce_orders ---
📊 Tamaño previo: 0.39 MB | Filas: 1000
✅ Ejecución exitosa.
📊 Tamaño optimizado: 0.15 MB
📉 Reducción de memoria: 61.95%
--- Inicio de Optimización en archivo: ecommerce_order_items ---
📊 Tamaño previo: 0.13 MB | Filas

In [322]:
for name, df in datasets.items():
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0]

    if not null_cols.empty:
        print(f"\n📂 {name}")
        print(null_cols.to_frame("nulos").assign(porcentaje=lambda x: (x["nulos"] / len(df) * 100).round(2)))


📂 ecommerce_categories
                    nulos  porcentaje
parent_category_id      7        70.0

📂 ecommerce_orders
              nulos  porcentaje
promotion_id    724        72.4
notes           837        83.7


In [323]:
#Verificamos que el ecommerce_categories tiene 7 filas con valores nulos en parent_category_id.

df_categories = datasets['ecommerce_categories']

# Ver exactamente cuáles son las filas con nulos en esa columna
df_categories[df_categories["parent_category_id"].isnull()]

,category_id,category_name,description,parent_category_id,is_active,display_order
0,1,Electrónica,Productos de electrónica,NaN,True,1
1,2,Ropa,Productos de ropa,NaN,True,2
2,3,Hogar,Productos de hogar,NaN,True,3
3,4,Deportes,Productos de deportes,NaN,True,4
6,7,Belleza,Productos de belleza,NaN,True,7
7,8,Alimentos,Productos de alimentos,NaN,False,8
8,9,Jardín,Productos de jardín,NaN,True,9


Estos valores no los modifico por que considero que son categorias que no tienen una raíz padre, lo cual puede ser correcto.

In [324]:
# Vamos a verificar el dataset ecommerce_orders que tiene varias filas con nulos

df_orders = datasets['ecommerce_orders']
df_orders[df_orders["promotion_id"].isnull()]
df_orders[df_orders["notes"].isnull()]
#No modificamos las ordenes con promotion_id null por que tal vez son ordenes que no tenían promoción o un precio de descuento al momento de hacer la compra.
#Tampoco modificamos campo notes.

,order_id,order_number,customer_id,order_date,status,subtotal,discount_percent,shipping_cost,tax_amount,total_amount,payment_method,shipping_method,promotion_id,notes
1,2,ORD-20230315-00002,21,2023-03-15,procesando,1379.520020,6,360,289.700012,1946.449951,paypal,same_day,10.0,NaN
2,3,ORD-20250807-00003,313,2025-08-07,enviado,1652.000000,16,30,346.920013,1764.599976,debit_card,express,NaN,NaN
3,4,ORD-20250811-00004,195,2025-08-11,procesando,1489.880005,2,300,312.869995,2072.949951,paypal,pickup,7.0,NaN
4,5,ORD-20251012-00005,257,2025-10-12,enviado,989.119995,6,470,207.720001,1607.489990,bank_transfer,same_day,NaN,NaN
5,6,ORD-20230126-00006,127,2023-01-26,cancelado,4588.899902,17,480,963.669983,5252.459961,paypal,pickup,NaN,NaN
6,7,ORD-20250716-00007,189,2025-07-16,cancelado,4813.660156,13,470,1010.869995,5668.750000,credit_card,same_day,NaN,NaN
8,9,ORD-20230122-00009,255,2023-01-22,entregado,784.710022,18,240,164.789993,1048.250000,debit_card,same_day,2.0,NaN
9,10,ORD-20240116-00010,56,2024-01-16,pendiente,2307.270020,4,230,484.529999,2929.510010,cash_on_delivery,express,NaN,NaN
10,11,ORD-20240818-00011,103,2024-08-18,enviado,3727.189941,3,420,782.710022,4818.080078,credit_card,same_day,NaN,NaN
11,12,ORD-20231003-00012,268,2023-10-03,procesando,3450.879883,2,10,724.679993,4116.540039,bank_transfer,standard,NaN,NaN


Ahora procedemos a verificar si hay valores repetidos en los datasets

In [325]:
# Diccionario para guardar solo los que tienen duplicados
dfs_con_duplicados = {}

for name, df in datasets.items():
    cantidad_duplicados = df.duplicated().sum()
    
    if cantidad_duplicados > 0:
        dfs_con_duplicados[name] = cantidad_duplicados
        print(f"⚠️  Archivo '{name}' tiene {cantidad_duplicados} filas duplicadas.")
    else:
        print(f"✅ Archivo '{name}' NO tiene duplicados.")

print(f"\nTotal de archivos a revisar: {len(dfs_con_duplicados)}")


✅ Archivo 'ecommerce_brands' NO tiene duplicados.
✅ Archivo 'ecommerce_categories' NO tiene duplicados.
✅ Archivo 'ecommerce_customers' NO tiene duplicados.
✅ Archivo 'ecommerce_inventory' NO tiene duplicados.
✅ Archivo 'ecommerce_orders' NO tiene duplicados.
✅ Archivo 'ecommerce_order_items' NO tiene duplicados.
✅ Archivo 'ecommerce_products' NO tiene duplicados.
✅ Archivo 'ecommerce_promotions' NO tiene duplicados.
✅ Archivo 'ecommerce_reviews' NO tiene duplicados.
✅ Archivo 'ecommerce_suppliers' NO tiene duplicados.
✅ Archivo 'ecommerce_warehouses' NO tiene duplicados.

Total de archivos a revisar: 0


Buscamos si hay algún tipo de dato a corregir

In [326]:
for name, df in datasets.items():
    print(f"Dataset: {name}")

Dataset: ecommerce_brands
Dataset: ecommerce_categories
Dataset: ecommerce_customers
Dataset: ecommerce_inventory
Dataset: ecommerce_orders
Dataset: ecommerce_order_items
Dataset: ecommerce_products
Dataset: ecommerce_promotions
Dataset: ecommerce_reviews
Dataset: ecommerce_suppliers
Dataset: ecommerce_warehouses


In [327]:
categories = datasets["ecommerce_categories"]
categories.info()
print(categories.dtypes)

categories.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   category_id         10 non-null     int8   
 1   category_name       10 non-null     str    
 2   description         10 non-null     str    
 3   parent_category_id  3 non-null      float32
 4   is_active           10 non-null     bool   
 5   display_order       10 non-null     int8   
dtypes: bool(1), float32(1), int8(2), str(2)
memory usage: 362.0 bytes
category_id              int8
category_name             str
description               str
parent_category_id    float32
is_active                bool
display_order            int8
dtype: object


,category_id,category_name,description,parent_category_id,is_active,display_order
0,1,Electrónica,Productos de electrónica,NaN,True,1
1,2,Ropa,Productos de ropa,NaN,True,2
2,3,Hogar,Productos de hogar,NaN,True,3
3,4,Deportes,Productos de deportes,NaN,True,4
4,5,Libros,Productos de libros,4.0,True,5
5,6,Juguetes,Productos de juguetes,5.0,True,6
6,7,Belleza,Productos de belleza,NaN,True,7
7,8,Alimentos,Productos de alimentos,NaN,False,8
8,9,Jardín,Productos de jardín,NaN,True,9
9,10,Automotriz,Productos de automotriz,4.0,True,10


Corregimos el tipo de dato parent_category_id de float a int32

In [328]:
categories["parent_category_id"] = categories["parent_category_id"].astype("Int32")
print(categories.dtypes)
categories.head(10)


category_id            int8
category_name           str
description             str
parent_category_id    Int32
is_active              bool
display_order          int8
dtype: object


,category_id,category_name,description,parent_category_id,is_active,display_order
0,1,Electrónica,Productos de electrónica,<NA>,True,1
1,2,Ropa,Productos de ropa,<NA>,True,2
2,3,Hogar,Productos de hogar,<NA>,True,3
3,4,Deportes,Productos de deportes,<NA>,True,4
4,5,Libros,Productos de libros,4,True,5
5,6,Juguetes,Productos de juguetes,5,True,6
6,7,Belleza,Productos de belleza,<NA>,True,7
7,8,Alimentos,Productos de alimentos,<NA>,False,8
8,9,Jardín,Productos de jardín,<NA>,True,9
9,10,Automotriz,Productos de automotriz,4,True,10


In [329]:
customers = datasets["ecommerce_customers"]
customers.info()
print(customers.dtypes)
customers.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 334 entries, 0 to 333
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   customer_id        334 non-null    int16   
 1   first_name         334 non-null    category
 2   last_name          334 non-null    category
 3   email              334 non-null    str     
 4   phone              334 non-null    str     
 5   birth_date         334 non-null    str     
 6   city               334 non-null    category
 7   country            334 non-null    category
 8   postal_code        334 non-null    int32   
 9   segment            334 non-null    category
 10  registration_date  334 non-null    str     
 11  last_login         334 non-null    str     
 12  is_verified        334 non-null    bool    
 13  accepts_marketing  334 non-null    int8    
dtypes: bool(1), category(5), int16(1), int32(1), int8(1), str(5)
memory usage: 18.1 KB
customer_id             int16
firs

,customer_id,first_name,last_name,email,phone,birth_date,city,country,postal_code,segment,registration_date,last_login,is_verified,accepts_marketing
0,1,Pedro,Fernández,pedro.fernández1@email.com,+1 855-934-2501,2005-01-26,Córdoba,Argentina,12469,Silver,2023-01-10,2024-07-10,True,0
1,2,Diego,López,diego.lópez2@email.com,+1 592-110-4122,1968-03-30,Tucumán,Argentina,28567,Platinum,2024-02-15,2024-07-02,True,1
2,3,Ana,Fernández,ana.fernández3@email.com,+1 645-895-5732,1986-11-19,Buenos Aires,Chile,37948,Bronze,2021-01-31,2022-08-04,True,1
3,4,Laura,Fernández,laura.fernández4@email.com,+1 724-383-4643,1998-01-20,Buenos Aires,México,71273,Silver,2021-10-22,2022-04-11,False,0
4,5,Lucía,Ramírez,lucía.ramírez5@email.com,+1 237-627-5106,1981-07-13,Mar del Plata,Argentina,37986,Silver,2022-08-08,2024-10-18,False,1
5,6,Lucas,López,lucas.lópez6@email.com,+1 521-510-6518,1965-02-02,Mendoza,Argentina,28800,Bronze,2020-06-09,2020-09-18,True,0
6,7,Juan,Sánchez,juan.sánchez7@email.com,+1 357-696-3958,1987-03-15,Buenos Aires,Argentina,70612,Platinum,2021-02-15,2023-10-20,True,1
7,8,Mateo,González,mateo.gonzález8@email.com,+1 401-783-4847,1964-07-05,Mendoza,Colombia,69822,Bronze,2020-10-30,2023-03-14,False,1
8,9,Lucas,López,lucas.lópez9@email.com,+1 421-762-8839,1976-06-28,Rosario,Colombia,52532,Gold,2023-09-04,2024-09-01,False,1
9,10,Camila,González,camila.gonzález10@email.com,+1 369-902-3292,1975-01-10,Rosario,Colombia,96513,Bronze,2022-01-13,2023-06-27,True,1


Modificamos algunos campos del dataset customers
first_name y last_name a str
birth_date, registration_date y last_login a datetime
accepts_marketing de int8 a booleano

In [330]:
customers = transform_customers(customers)
print(customers.dtypes)
customers.head(10)

customer_id                   int16
first_name                   string
last_name                    string
email                           str
phone                           str
birth_date           datetime64[us]
city                       category
country                    category
postal_code                   int32
segment                    category
registration_date    datetime64[us]
last_login           datetime64[us]
is_verified                    bool
accepts_marketing              bool
dtype: object


,customer_id,first_name,last_name,email,phone,birth_date,city,country,postal_code,segment,registration_date,last_login,is_verified,accepts_marketing
0,1,Pedro,Fernández,pedro.fernández1@email.com,+1 855-934-2501,2005-01-26,Córdoba,Argentina,12469,Silver,2023-01-10,2024-07-10,True,False
1,2,Diego,López,diego.lópez2@email.com,+1 592-110-4122,1968-03-30,Tucumán,Argentina,28567,Platinum,2024-02-15,2024-07-02,True,True
2,3,Ana,Fernández,ana.fernández3@email.com,+1 645-895-5732,1986-11-19,Buenos Aires,Chile,37948,Bronze,2021-01-31,2022-08-04,True,True
3,4,Laura,Fernández,laura.fernández4@email.com,+1 724-383-4643,1998-01-20,Buenos Aires,México,71273,Silver,2021-10-22,2022-04-11,False,False
4,5,Lucía,Ramírez,lucía.ramírez5@email.com,+1 237-627-5106,1981-07-13,Mar del Plata,Argentina,37986,Silver,2022-08-08,2024-10-18,False,True
5,6,Lucas,López,lucas.lópez6@email.com,+1 521-510-6518,1965-02-02,Mendoza,Argentina,28800,Bronze,2020-06-09,2020-09-18,True,False
6,7,Juan,Sánchez,juan.sánchez7@email.com,+1 357-696-3958,1987-03-15,Buenos Aires,Argentina,70612,Platinum,2021-02-15,2023-10-20,True,True
7,8,Mateo,González,mateo.gonzález8@email.com,+1 401-783-4847,1964-07-05,Mendoza,Colombia,69822,Bronze,2020-10-30,2023-03-14,False,True
8,9,Lucas,López,lucas.lópez9@email.com,+1 421-762-8839,1976-06-28,Rosario,Colombia,52532,Gold,2023-09-04,2024-09-01,False,True
9,10,Camila,González,camila.gonzález10@email.com,+1 369-902-3292,1975-01-10,Rosario,Colombia,96513,Bronze,2022-01-13,2023-06-27,True,True


In [331]:
inventory = datasets["ecommerce_inventory"]

inventory = transform_inventory(inventory)
print(inventory.dtypes)


inventory_id                  int16
product_id                     int8
warehouse_id                   int8
quantity                      int16
min_stock_level                int8
max_stock_level               int16
last_restock_date    datetime64[us]
dtype: object


In [332]:
orders = datasets["ecommerce_orders"]
print(orders.dtypes)
orders.head(5)

order_id               int16
order_number             str
customer_id            int16
order_date               str
status              category
subtotal             float32
discount_percent        int8
shipping_cost          int16
tax_amount           float32
total_amount         float32
payment_method      category
shipping_method     category
promotion_id         float32
notes               category
dtype: object


,order_id,order_number,customer_id,order_date,status,subtotal,discount_percent,shipping_cost,tax_amount,total_amount,payment_method,shipping_method,promotion_id,notes
0,1,ORD-20230131-00001,131,2023-01-31,enviado,2923.739990,6,440,613.989990,3802.310059,paypal,same_day,NaN,Nota del cliente: Envolver para regalo
1,2,ORD-20230315-00002,21,2023-03-15,procesando,1379.520020,6,360,289.700012,1946.449951,paypal,same_day,10.0,NaN
2,3,ORD-20250807-00003,313,2025-08-07,enviado,1652.000000,16,30,346.920013,1764.599976,debit_card,express,NaN,NaN
3,4,ORD-20250811-00004,195,2025-08-11,procesando,1489.880005,2,300,312.869995,2072.949951,paypal,pickup,7.0,NaN
4,5,ORD-20251012-00005,257,2025-10-12,enviado,989.119995,6,470,207.720001,1607.489990,bank_transfer,same_day,NaN,NaN


Modificamos algunos campos del dataset orders
order_date a datetime
promotion_id de float a int
notes de category a string

In [333]:
orders = transform_orders(orders)
print(orders.dtypes)

order_id                     int16
order_number                   str
customer_id                  int16
order_date          datetime64[us]
status                    category
subtotal                   float32
discount_percent              int8
shipping_cost                int16
tax_amount                 float32
total_amount               float32
payment_method            category
shipping_method           category
promotion_id                  Int8
notes                       string
dtype: object


In [334]:
products = datasets["ecommerce_products"]
print(products.dtypes)

products.head(5)

product_id         int8
sku                 str
product_name        str
description         str
category_id        int8
brand_id           int8
supplier_id        int8
price           float32
cost            float32
weight_kg       float32
is_active          bool
created_at          str
updated_at          str
dtype: object


,product_id,sku,product_name,description,category_id,brand_id,supplier_id,price,cost,weight_kg,is_active,created_at,updated_at
0,1,SKU-000001,Elite Kit 1,Descripción detallada del producto 1,4,2,4,260.570007,148.520004,32.900002,True,2023-07-16,2026-02-21
1,2,SKU-000002,Mini Producto 2,Descripción detallada del producto 2,8,3,6,135.970001,81.580002,16.500000,False,2022-07-31,2024-09-26
2,3,SKU-000003,Elite Set 3,Descripción detallada del producto 3,5,15,5,301.309998,183.800003,34.400002,True,2022-03-10,2024-11-01
3,4,SKU-000004,Mega Equipo 4,Descripción detallada del producto 4,9,7,8,32.340000,14.880000,37.900002,True,2023-07-24,2025-06-14
4,5,SKU-000005,Max Equipo 5,Descripción detallada del producto 5,8,10,3,478.839996,258.570007,14.400000,True,2023-04-03,2024-09-20


Modificamos los campos created_at y updated_at de str a datetime en ecommerce_products

In [335]:
products = transform_products(products)
print(products.dtypes)

product_id                int8
sku                        str
product_name               str
description                str
category_id               int8
brand_id                  int8
supplier_id               int8
price                  float32
cost                   float32
weight_kg              float32
is_active                 bool
created_at      datetime64[us]
updated_at      datetime64[us]
dtype: object


In [336]:
promotions = datasets["ecommerce_promotions"]
print(promotions.dtypes)
promotions.head(3)

promotion_id            int8
promotion_code           str
promotion_name           str
promotion_type      category
discount_value          int8
min_order_amount       int16
max_uses               int16
current_uses           int16
start_date               str
end_date                 str
is_active               bool
dtype: object


,promotion_id,promotion_code,promotion_name,promotion_type,discount_value,min_order_amount,max_uses,current_uses,start_date,end_date,is_active
0,1,PROMO0EED47,Promoción 1,percentage,8,170,638,406,2024-04-27,2024-07-11,True
1,2,PROMO80EB43,Promoción 2,fixed_amount,32,840,6085,491,2024-01-27,2024-03-15,True
2,3,PROMO7F4F39,Promoción 3,percentage,30,630,8128,15,2024-01-10,2024-03-18,False


Modificamos los campos fecha de ecommerce_promotions

In [337]:
promotions = transform_promotions(promotions)
print(promotions.dtypes)

promotion_id                  int8
promotion_code                 str
promotion_name                 str
promotion_type            category
discount_value                int8
min_order_amount             int16
max_uses                     int16
current_uses                 int16
start_date          datetime64[us]
end_date            datetime64[us]
is_active                     bool
dtype: object


In [338]:
reviews = datasets["ecommerce_reviews"]
print(reviews.dtypes)
reviews.head(5)

review_id                  int16
product_id                  int8
customer_id                int16
rating                      int8
title                   category
comment                 category
is_verified_purchase        bool
helpful_votes               int8
created_at                   str
dtype: object


,review_id,product_id,customer_id,rating,title,comment,is_verified_purchase,helpful_votes,created_at
0,1,77,229,5,Podría mejorar,Cumple con lo prometido.,False,44,2023-12-16
1,2,28,119,5,Buena calidad,Cumple con lo prometido.,True,45,2024-01-06
2,3,16,75,3,No tan bueno,"Superó mis expectativas, lo recomiendo totalme...",True,47,2024-02-26
3,4,26,244,3,Buena calidad,Cumple con lo prometido.,False,16,2023-10-09
4,5,91,205,1,Buena calidad,Llegó a tiempo y en perfecto estado.,True,46,2023-04-17


In [339]:
reviews = transform_reviews(reviews)
print(reviews.dtypes)

review_id                        int16
product_id                        int8
customer_id                      int16
rating                            int8
title                         category
comment                       category
is_verified_purchase              bool
helpful_votes                     int8
created_at              datetime64[us]
dtype: object


In [340]:
suppliers = datasets["ecommerce_suppliers"]
print(suppliers.dtypes)

suppliers.head(5)

supplier_id         int8
supplier_name        str
contact_name         str
email                str
phone                str
address              str
rating           float32
is_active           bool
dtype: object


,supplier_id,supplier_name,contact_name,email,phone,address,rating,is_active
0,1,Distribuidora Norte,Juan Pérez,contact@distribuidoranorte.com,+1 997-115-5178,2043 Rosario,4.2,False
1,2,Importadora Sur,Ana Martínez,contact@importadorasur.com,+1 989-643-6232,2931 La Plata,4.2,True
2,3,Mayorista Central,María García,contact@mayoristacentral.com,+1 934-650-7368,7595 Córdoba,3.8,True
3,4,Logística Express,Carlos López,contact@logísticaexpress.com,+1 362-407-3027,2420 Mar del Plata,3.5,True
4,5,Proveedores Unidos,Carlos López,contact@proveedoresunidos.com,+1 448-550-2292,1861 Santa Fe,4.1,True


In [341]:
warehouses = datasets["ecommerce_warehouses"]
print(warehouses.dtypes)

warehouses.head(5)

warehouse_id          int8
warehouse_name         str
location               str
capacity_units       int32
current_occupancy     int8
manager_name           str
dtype: object


,warehouse_id,warehouse_name,location,capacity_units,current_occupancy,manager_name
0,1,Depósito 1,Buenos Aires,50548,72,Diego Fernández
1,2,Depósito 2,Córdoba,37177,85,Laura Torres
2,3,Depósito 3,Rosario,52044,48,Sofía Ramírez
3,4,Depósito 4,Mendoza,47436,83,Sofía Ramírez
4,5,Depósito 5,La Plata,11129,55,Diego Fernández


1.Cuales son los 5 clientes que más dinero gastaron ?

In [342]:

top_five_clients = (
    pd.merge(orders, customers, on="customer_id")
    .query("status == 'completed'")
    .groupby(["customer_id", "first_name", "last_name"])
    .agg(
        total_orders=("order_id", "count"), 
        total_spent=("total_amount", "sum"))
        .sort_values(by="total_spent", ascending=False).reset_index().head(5)
)

print(top_five_clients)

Empty DataFrame
Columns: [customer_id, first_name, last_name, total_orders, total_spent]
Index: []


2.Cuál es el producto más vendido (por cantidad)

In [343]:
order_items = datasets["ecommerce_order_items"]

most_selled_product = pd.merge(order_items, products, on="product_id").groupby(["product_id", "product_name"]).agg(total_sold=("quantity", "sum")).sort_values(by="total_sold", ascending=False).reset_index().head(1)
print(most_selled_product)

   product_id    product_name  total_sold
0          44  Mega Equipo 44         126


3.Como evolucionaron las ventas mes a mes

In [344]:
orders.head(1)

,order_id,order_number,customer_id,order_date,status,subtotal,discount_percent,shipping_cost,tax_amount,total_amount,payment_method,shipping_method,promotion_id,notes
0,1,ORD-20230131-00001,131,2023-01-31,enviado,2923.73999,6,440,613.98999,3802.310059,paypal,same_day,<NA>,Nota del cliente: Envolver para regalo


In [345]:
sales_by_month = (
    orders
    .query("status != 'cancelado'")
    .groupby([orders["order_date"].dt.year.rename("año"), orders["order_date"].dt.month.rename("mes")])
    .agg(
        total_ventas=("total_amount", "sum"),
    )
)

sales_by_month["porcentaje_crecimiento"] = sales_by_month["total_ventas"].pct_change().round(2).fillna(0) * 100

print(sales_by_month)

          total_ventas  porcentaje_crecimiento
año  mes                                      
2023 1    37909.710938                     0.0
     2    68137.460938                    80.0
     3    39488.878906                   -42.0
     4    58773.531250                    49.0
     5    57834.179688                    -2.0
     6    46536.031250                   -20.0
     7    48476.519531                     4.0
     8    57348.281250                    18.0
     9    30737.830078                   -46.0
     10   59746.730469                    94.0
     11   84217.406250                    41.0
     12   70024.890625                   -17.0
2024 1    66097.484375                    -6.0
     2    71261.921875                     8.0
     3    67306.843750                    -6.0
     4    63494.257812                    -6.0
     5    58305.039062                    -8.0
     6    64488.109375                    11.0
     7    66225.359375                     3.0
     8    662

Mostramos el total de ventas mes a mes con su porcentaje de crecimiento

Procedemos a guardar en CSV y Parquet

In [ ]:
for name, df in datasets.items():
    load_datasets